#### J-function Development

In [22]:
%load_ext autoreload
%autoreload 2
import numpy as np
from scipy.spatial import KDTree
import pandas as pd
import matplotlib.pyplot as plt
from ecd_helperFunctions import *

def j_function(points_type1, points_type2, bbox, r_max, dr):
    """
    Calculates the J-function between two types of points in a 2D spatial distribution.

    Parameters:
    -----------
    points_type1 : array_like
        An (N1, 2) array of coordinates for points of Type 1.
    points_type2 : array_like
        An (N2, 2) array of coordinates for points of Type 2.
    bbox : tuple
        A tuple defining the study area's bounds: (x_min, x_max, y_min, y_max).
    r_max : float
        Maximum radius to compute the functions.
    dr : float
        Width of the radial distance bins.

    Returns:
    --------
    r : numpy.ndarray
        Array of radial distance bin centers.
    J_r : numpy.ndarray
        J-function values for each bin.
    """
    # Ensure inputs are NumPy arrays
    points_type1 = np.asarray(points_type1)
    points_type2 = np.asarray(points_type2)

    if len(points_type1) == 0 or len(points_type2) == 0:
        return 0, 0

    # Build KDTree for efficient neighbor searches
    tree_type1 = KDTree(points_type1)
    tree_type2 = KDTree(points_type2)

    # Set up radial bins
    r_edges = np.arange(0, r_max + dr, dr)
    r_centers = (r_edges[:-1] + r_edges[1:]) / 2

    # Calculate G12(r): from Type 1 to nearest Type 2
    distances, _ = tree_type1.query(points_type2, k=1)
    G12_counts, _ = np.histogram(distances, bins=r_edges)
    G12_cumulative = np.cumsum(G12_counts) / len(points_type2)

    # Calculate F12(r): from random points to nearest Type 2
    # Generate random points within the study area
    num_random_points = 1000  # Adjust as needed for accuracy
    x_min, x_max, y_min, y_max = bbox
    random_x = np.random.uniform(x_min, x_max, num_random_points)
    random_y = np.random.uniform(y_min, y_max, num_random_points)
    random_points = np.column_stack((random_x, random_y))

    # Compute distances from random points to nearest Type 2 point
    distances_random, _ = tree_type2.query(random_points, k=1)
    F12_counts, _ = np.histogram(distances_random, bins=r_edges)
    F12_cumulative = np.cumsum(F12_counts) / num_random_points

    # Compute J-function
    # Avoid division by zero by adding a small epsilon
    epsilon = 1e-10
    J_r = (1 - G12_cumulative + epsilon) / (1 - F12_cumulative + epsilon)

    return r_centers, J_r


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
def calculate_j_function(data_path=None,
                 df=None,
                 distance_cols=['x_centroid', 'y_centroid'],
                 cell_a_label=0,
                 cell_b_label=1,
                 return_bboxes=False,
                 perturb_matrix=False,
                 roi_tile_size=1000,
                 cell_label_col='tumor_immune',
                 r_max=300,
                 dr=5):
    """
    Calculate the Getis-Ord G* statistic for spatial point patterns.

    This function calculates the Getis-Ord G* statistic to identify spatial clusters of high or low values
    in point pattern data. It can analyze both distance-based and intensity-based patterns.

    Parameters:
        data_path (str, optional): Path to CSV file containing point pattern data. Default is None.
        df (pandas.DataFrame, optional): DataFrame containing point pattern data. Default is None.
        distance_cols (list): Column names for spatial coordinates. Default is ['x_centroid', 'y_centroid'].
        intensity_cols (list): Column names for intensity values. Default is ['sox2_mean', 'cd45_mean'].
        perturb_matrix (bool): Whether to perturb the distance matrix. Default is False.
        roi_tile_size (int): Size of regions of interest tiles. Default is 1000.
        cell_label_col (str): Column name for cell type labels. Default is 'tumor_immune'.
        return_bboxes (bool): Whether to return bounding boxes. Default is False.

    Returns:
        list: Getis-Ord G* statistics for each region
        dict: Cell counts per region if return_bboxes is True

    Raises:
        ValueError: If neither data_path nor df is provided
    """
    j_functions = []
    if data_path is not None:
        df = pd.read_csv(data_path)
    elif df is not None:
        pass
    else:
        raise ValueError("No data path or dataframe provided")

    # Divide data into ROIs
    roi_data, bboxes = create_tile_rois(df, 
                                 tile_size=roi_tile_size, 
                                 x_coord=distance_cols[0], 
                                 y_coord=distance_cols[1],
                                 return_bboxes=True)

    if perturb_matrix is True:
        j_function_dict = {}
        for i, key in enumerate(roi_data.keys()):
            print("Calculating perturbed intensity matrix for ROI: ", key, "for ", i, " of ", len(roi_data.keys()), " ROIs")
            roi_data[key] = spatial_perturb_matrix(roi_data[key], perturb_columns=distance_cols)
            cell_a_points = roi_data[key][roi_data[key][cell_label_col] == cell_a_label][distance_cols].values
            cell_b_points = roi_data[key][roi_data[key][cell_label_col] == cell_b_label][distance_cols].values
            val_j_function, _ = j_function(cell_a_points, cell_b_points,  bboxes[key], r_max=r_max, dr=dr)
            j_functions.append(val_j_function)
            j_function_dict[key] = val_j_function

    return j_functions, j_function_dict

In [29]:
# Load in CSV as dataframe
sample_df = pd.read_csv('/home/ecdyer/labshare/PROJECTS/SPATIAL_STATS/data/P2_BTC/S2.csv')

j_functions, j_function_dict = calculate_j_function(df=sample_df,
                 distance_cols=['x_centroid', 'y_centroid'],
                 cell_a_label=0,
                 cell_b_label=1,
                 perturb_matrix=True,
                 roi_tile_size=500,
                 cell_label_col='tumor_immune',
                 r_max=300,
                 dr=5,
                 return_bboxes=True)

print(j_function_dict)

Calculating perturbed intensity matrix for ROI:  (63, 5) for  0  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (63, 6) for  1  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (63, 7) for  2  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (63, 8) for  3  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (64, 4) for  4  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (64, 5) for  5  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (64, 6) for  6  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (64, 7) for  7  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (64, 8) for  8  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (65, 3) for  9  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (65, 4) for  10  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (65, 5) for  11  of  23  ROIs
Calculating perturbed intensity matrix for ROI:  (65, 6) for  